In [1]:
import torch
import os
# Enable expandable segments to prevent memory fragmentation
os.environ['PYTORCH_ALLOC_CONF'] = 'expandable_segments:True'
print(torch.__version__)        # e.g. 2.1.0
print(torch.cuda.is_available()) # must be True

2.10.0+cu128
True


In [2]:
!pip install ninja packaging
!pip install causal-conv1d --no-build-isolation
!pip install mamba-ssm --no-build-isolation

  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.2/12.2 MB 101.7 MB/s eta 0:00:0000:01:01
  Created wheel for causal-conv1d: filename=causal_conv1d-1.6.2.post1-cp312-cp312-linux_x86_64.whl size=193835395 sha256=c16c1c48d4fa63415cc797e02d69f97248c57c04627d99e394d5bb0ef266e288
  Stored in directory: /root/.cache/pip/wheels/ea/f0/26/5d87ae05a302e6dc8016c50cd8c7ee779585f593b9580e7cf8
Successfully built causal-conv1d
  Attempting uninstall: cuda-bindings
    Found existing installation: cuda-bindings 13.2.0
    Uninstalling cuda-bindings-13.2.0:
      Successfully uninstalled cuda-bindings-13.2.0
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
dask-cuda 26.2.0 requires cuda-core==0.3.*, but you have cuda-core 1.0.1 which is incompatible.
dask-cuda 26.2.0 requires numba-cuda<0.23.0,>=0.22.1, but you have numba-cuda 0.30.2 

In [3]:
import mamba_ssm
print(mamba_ssm.__version__)  # should print 2.3.2.post1

2.3.2.post1


In [4]:
!wget -q https://ftp.ncbi.nlm.nih.gov/pub/clinvar/tab_delimited/variant_summary.txt.gz
!gunzip variant_summary.txt.gz

In [5]:
import pandas as pd

df = pd.read_csv('variant_summary.txt', sep='\t', low_memory=False)
print(df.shape)
print(df['ClinicalSignificance'].value_counts())

(8982609, 43)
ClinicalSignificance
Uncertain significance                                  4673771
Likely benign                                           2182660
-                                                        490980
Benign                                                   425638
Pathogenic                                               404641
                                                         ...   
Benign; Affects; association; other                           2
Likely benign; association                                    2
Likely pathogenic; protective                                 2
Pathogenic/Likely pathogenic/Established risk allele          2
Established risk allele; association                          2
Name: count, Length: 105, dtype: int64


In [6]:
# Check exact column names
print("COLUMNS:")
print(df.columns.tolist())

print("\nASSEMBLY values:")
print(df['Assembly'].value_counts().head())

print("\nTYPE values:")
print(df['Type'].value_counts().head())

print("\nREF sample values:")
print(df['ReferenceAllele'].value_counts().head())

print("\nALT sample values:")
print(df['AlternateAllele'].value_counts().head())

COLUMNS:
['#AlleleID', 'Type', 'Name', 'GeneID', 'GeneSymbol', 'HGNC_ID', 'ClinicalSignificance', 'ClinSigSimple', 'LastEvaluated', 'RS# (dbSNP)', 'nsv/esv (dbVar)', 'RCVaccession', 'PhenotypeIDS', 'PhenotypeList', 'Origin', 'OriginSimple', 'Assembly', 'ChromosomeAccession', 'Chromosome', 'Start', 'Stop', 'ReferenceAllele', 'AlternateAllele', 'Cytogenetic', 'ReviewStatus', 'NumberSubmitters', 'Guidelines', 'TestedInGTR', 'OtherIDs', 'SubmitterCategories', 'VariationID', 'PositionVCF', 'ReferenceAlleleVCF', 'AlternateAlleleVCF', 'SomaticClinicalImpact', 'SomaticClinicalImpactLastEvaluated', 'ReviewStatusClinicalImpact', 'Oncogenicity', 'OncogenicityLastEvaluated', 'ReviewStatusOncogenicity', 'SCVsForAggregateGermlineClassification', 'SCVsForAggregateSomaticClinicalImpact', 'SCVsForAggregateOncogenicityClassification']

ASSEMBLY values:
Assembly
GRCh37    4510851
GRCh38    4457595
na           9392
NCBI36       4771
Name: count, dtype: int64

TYPE values:
Type
single nucleotide variant  

In [7]:
for col in ['Assembly', 'Type', 'ReferenceAlleleVCF', 'AlternateAlleleVCF', 'ClinicalSignificance']:
    df[col] = df[col].str.strip()

label_map = {
    'Benign': 'Benign',
    'Likely benign': 'Likely Benign',
    'Uncertain significance': 'VUS',
    'Likely pathogenic': 'Likely Pathogenic',
    'Pathogenic': 'Pathogenic',
}

df_clean = df[
    (df['Assembly'] == 'GRCh38') &
    (df['Type'] == 'single nucleotide variant') &
    (df['ClinicalSignificance'].isin(label_map.keys())) &
    (df['ReferenceAlleleVCF'].isin(['A','T','G','C'])) &
    (df['AlternateAlleleVCF'].isin(['A','T','G','C']))
].copy()

df_clean['label'] = df_clean['ClinicalSignificance'].map(label_map)

print(df_clean.shape)
print(df_clean['label'].value_counts())

(3641842, 44)
label
VUS                  2252331
Likely Benign        1048211
Benign                180374
Pathogenic             84594
Likely Pathogenic      76332
Name: count, dtype: int64


In [8]:
# Har class se 50,000 samples lenge
SAMPLES_PER_CLASS = 50_000

df_balanced = df_clean.groupby('label').apply(
    lambda x: x.sample(min(len(x), SAMPLES_PER_CLASS), random_state=42)
).reset_index(drop=True)

# Shuffle
df_balanced = df_balanced.sample(frac=1, random_state=42).reset_index(drop=True)

print(df_balanced.shape)
print(df_balanced['label'].value_counts())

/tmp/ipykernel_58/2312059223.py:4: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  df_balanced = df_clean.groupby('label').apply(


(250000, 44)
label
Benign               50000
Likely Benign        50000
Likely Pathogenic    50000
Pathogenic           50000
VUS                  50000
Name: count, dtype: int64


In [9]:
!pip install pyfaidx -q

# Download hg38 - sirf chromosomes 1-22, X, Y (full genome ~900MB)
!wget -q --show-progress https://hgdownload.soe.ucsc.edu/goldenPath/hg38/bigZips/hg38.fa.gz
!gunzip hg38.fa.gz

hg38.fa.gz          100%[===================>] 938.09M  39.6MB/s    in 22s     


In [10]:
from pyfaidx import Fasta
import pandas as pd

# Load reference genome
genome = Fasta('hg38.fa')

print("Genome loaded!")
print("Chromosomes available:", list(genome.keys())[:5], "...")

Genome loaded!
Chromosomes available: ['chr1', 'chr10', 'chr11', 'chr11_KI270721v1_random', 'chr12'] ...


In [11]:
def get_sequence(row, genome, window=512):
    try:
        chrom = 'chr' + str(row['Chromosome'])
        pos = int(row['PositionVCF']) - 1  # 0-based index
        
        # Window ke start aur end
        start = max(0, pos - window)
        end = pos + window
        
        # Reference sequence fetch karo
        seq = str(genome[chrom][start:end])
        
        # REF base ko ALT se replace karo (mutation insert karo)
        mid = pos - start
        seq = seq[:mid] + row['AlternateAlleleVCF'] + seq[mid+1:]
        
        return seq.upper()
    except:
        return None

# Test on 1 row first
sample = df_balanced.iloc[0]
seq = get_sequence(sample, genome)
print("Sample sequence:")
print(seq[:50], "...")
print("Length:", len(seq))

Sample sequence:
GCCCGCCTTGGCCTCCCAAAGTGCTGGGATTACAGGCGTGAGCCACTGTG ...
Length: 1024


In [12]:
from tqdm import tqdm
tqdm.pandas()

print("Fetching sequences for all 250,000 variants...")
df_balanced['sequence'] = df_balanced.progress_apply(
    lambda row: get_sequence(row, genome), axis=1
)

# Drop failed rows
df_balanced = df_balanced.dropna(subset=['sequence'])

print(f"\nDone! {len(df_balanced)} sequences fetched")
print(df_balanced['label'].value_counts())

Fetching sequences for all 250,000 variants...


100%|██████████| 250000/250000 [00:06<00:00, 36284.48it/s]



Done! 249632 sequences fetched
label
Pathogenic           49979
VUS                  49976
Likely Benign        49965
Likely Pathogenic    49946
Benign               49766
Name: count, dtype: int64


In [13]:
# Save kar lo abhi
df_balanced[['sequence', 'label']].to_csv('clinvar_sequences.csv', index=False)
print("Saved!")

Saved!


In [14]:
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained(
    "kuleshov-group/caduceus-ps_seqlen-131k_d_model-256_n_layer-16",
    trust_remote_code=True
)

# Test on one sequence
test_seq = df_balanced['sequence'].iloc[0]
tokens = tokenizer(test_seq, return_tensors='pt')
print("Token shape:", tokens['input_ids'].shape)
print("Sample tokens:", tokens['input_ids'][0][:10])

config.json: 0.00B [00:00, ?B/s]

configuration_caduceus.py: 0.00B [00:00, ?B/s]

A new version of the following files was downloaded from https://huggingface.co/kuleshov-group/caduceus-ps_seqlen-131k_d_model-256_n_layer-16:
- configuration_caduceus.py
. Make sure to double-check they do not contain any added malicious code. To avoid downloading new versions of the code file, you can pin a revision.


tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenization_caduceus.py: 0.00B [00:00, ?B/s]

A new version of the following files was downloaded from https://huggingface.co/kuleshov-group/caduceus-ps_seqlen-131k_d_model-256_n_layer-16:
- tokenization_caduceus.py
. Make sure to double-check they do not contain any added malicious code. To avoid downloading new versions of the code file, you can pin a revision.


special_tokens_map.json:   0%|          | 0.00/173 [00:00<?, ?B/s]

Token shape: torch.Size([1, 1025])
Sample tokens: tensor([ 9,  8,  8,  8,  9,  8,  8, 10, 10,  9])


In [15]:
!git clone https://huggingface.co/kuleshov-group/caduceus-ps_seqlen-131k_d_model-256_n_layer-16 caduceus_model

Cloning into 'caduceus_model'...
remote: Enumerating objects: 49, done.
remote: Total 49 (delta 0), reused 0 (delta 0), pack-reused 49 (from 1)
Receiving objects: 100% (49/49), 36.61 KiB | 5.23 MiB/s, done.
Resolving deltas: 100% (20/20), done.


In [16]:
import os

base = '/kaggle/working/caduceus_model'

# modeling_caduceus.py mein relative imports fix karo
with open(f'{base}/modeling_caduceus.py', 'r') as f:
    content = f.read()

content = content.replace('from .configuration_caduceus', 'from configuration_caduceus')
content = content.replace('from .modeling_rcps', 'from modeling_rcps')

with open(f'{base}/modeling_caduceus.py', 'w') as f:
    f.write(content)

# modeling_rcps.py mein bhi fix karo
with open(f'{base}/modeling_rcps.py', 'r') as f:
    content = f.read()

content = content.replace('from .configuration_caduceus', 'from configuration_caduceus')

with open(f'{base}/modeling_rcps.py', 'w') as f:
    f.write(content)

print("Imports patched!")

Imports patched!


In [17]:
import sys
sys.path.insert(0, '/kaggle/working/caduceus_model')

import importlib.util

def load_module(name, path):
    spec = importlib.util.spec_from_file_location(name, path)
    module = importlib.util.module_from_spec(spec)
    sys.modules[name] = module
    spec.loader.exec_module(module)
    return module

base = '/kaggle/working/caduceus_model'

# Sahi order mein load karo
load_module('configuration_caduceus', f'{base}/configuration_caduceus.py')
load_module('modeling_rcps',          f'{base}/modeling_rcps.py')
load_module('modeling_caduceus',      f'{base}/modeling_caduceus.py')
load_module('tokenization_caduceus',  f'{base}/tokenization_caduceus.py')

from configuration_caduceus import CaduceusConfig
from modeling_caduceus import Caduceus
from tokenization_caduceus import CaduceusTokenizer

# Config aur tokenizer load karo
config    = CaduceusConfig.from_pretrained(base)
tokenizer = CaduceusTokenizer.from_pretrained(base)

print("Config d_model:", config.d_model)
print("Vocab size:", config.vocab_size)

# Test tokenizer
seq = "ATGCATGCATGC"
tokens = tokenizer(seq, return_tensors='pt')
print("Token shape:", tokens['input_ids'].shape)
print("All good!")

Config d_model: 256
Vocab size: 16
Token shape: torch.Size([1, 13])
All good!


In [18]:
!pip install safetensors

In [19]:
import torch
import torch.nn as nn
from safetensors.torch import load_file

class CaduceusClassifier(nn.Module):
    def __init__(self, config, num_labels=5, base_path="/kaggle/working/caduceus_model"):
        super().__init__()
        self.encoder = Caduceus(config)
        
        # 1. Load weights
        weights = load_file(f'/kaggle/working/caduceus_model/model.safetensors', device='cpu')
        new_weights = {k.replace('caduceus.', ''): v for k, v in weights.items()}
        self.encoder.load_state_dict(new_weights, strict=False)
        print("✅ Weights loaded successfully!")
        
        # 🚀 2. FIX: Calculate hidden dimension mathematically (No dummy forward pass!)
        # Caduceus is a bidirectional model (processes forward + reverse complement).
        # Therefore, its final pooled output dimension is exactly 2 * d_model.
        hidden_dim = config.d_model * 2 
        print(f"📏 Encoder Output Dimension: {hidden_dim} (Calculated as d_model * 2)")
        
        # 3. Build the classifier with the CORRECT input size (512 -> 256 -> 5)
        self.classifier = nn.Sequential(
            nn.Linear(hidden_dim, 256), 
            nn.ReLU(),
            nn.Dropout(0.1),
            nn.Linear(256, num_labels)
        )

    def forward(self, input_ids, labels=None):
        outputs = self.encoder(input_ids=input_ids)
        
        # Handle both HuggingFace ModelOutput objects and raw tuples
        if hasattr(outputs, 'last_hidden_state'):
            pooled = outputs.last_hidden_state.mean(dim=1)
        else:
            pooled = outputs[0].mean(dim=1)
            
        logits = self.classifier(pooled)
        loss = None
        if labels is not None:
            loss = nn.CrossEntropyLoss()(logits, labels)
            
        return {'loss': loss, 'logits': logits}

# Re-initialize the model
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model = CaduceusClassifier(config=config, num_labels=5).to(device)
print(f"✅ Model initialized on {device}")
print(f"Total parameters: {sum(p.numel() for p in model.parameters()):,}")

✅ Weights loaded successfully!
📏 Encoder Output Dimension: 512 (Calculated as d_model * 2)
✅ Model initialized on cuda
Total parameters: 7,857,925


In [20]:
import torch
import pandas as pd
from tqdm import tqdm

print("Pre-tokenizing all sequences...")
df = pd.read_csv('clinvar_sequences.csv')

all_input_ids = []
all_labels = []

label_map = {'Benign': 0, 'Likely Benign': 1, 'VUS': 2, 'Likely Pathogenic': 3, 'Pathogenic': 4}

for idx, row in tqdm(df.iterrows(), total=len(df)):
    encoding = tokenizer(
        row['sequence'],
        truncation=True,
        max_length=1024,
        padding='max_length',
        return_tensors='pt'
    )
    all_input_ids.append(encoding['input_ids'].squeeze(0))
    all_labels.append(label_map[row['label']])

# Save as torch tensors
torch.save({
    'input_ids': torch.stack(all_input_ids),
    'labels': torch.tensor(all_labels)
}, '/kaggle/working/pretokenized_data.pt')

print("✅ Saved pretokenized data!")

Pre-tokenizing all sequences...


100%|██████████| 249632/249632 [12:34<00:00, 330.98it/s]


✅ Saved pretokenized data!


In [21]:
from torch.utils.data import Dataset, DataLoader
from sklearn.model_selection import train_test_split

class FastClinVarDataset(Dataset):
    def __init__(self, input_ids, labels):
        self.input_ids = input_ids
        self.labels = labels

    def __len__(self):
        return len(self.input_ids)

    def __getitem__(self, idx):
        return {
            'input_ids': self.input_ids[idx],
            'labels': self.labels[idx]
        }

# Load pre-tokenized data
data = torch.load('/kaggle/working/pretokenized_data.pt')

# Train/Val Split
train_ids, val_ids, train_lbls, val_lbls = train_test_split(
    data['input_ids'], data['labels'], 
    test_size=0.1, random_state=42, stratify=data['labels']
)

# 🚀 Increased Batch Size to 32 since we have 2 GPUs (16 per GPU)
BATCH_SIZE = 16

train_loader = DataLoader(
    FastClinVarDataset(train_ids, train_lbls), 
    batch_size=BATCH_SIZE, shuffle=True, num_workers=2, pin_memory=True
)
val_loader = DataLoader(
    FastClinVarDataset(val_ids, val_lbls), 
    batch_size=BATCH_SIZE, shuffle=False, num_workers=2, pin_memory=True
)

print(f"Train batches: {len(train_loader)} | Val batches: {len(val_loader)}")

Train batches: 14042 | Val batches: 1561


In [22]:
import torch.nn as nn
from safetensors.torch import load_file

class CaduceusClassifier(nn.Module):
    def __init__(self, config, num_labels=5):
        super().__init__()
        self.encoder = Caduceus(config)
        
        # Load weights
        weights = load_file('/kaggle/working/caduceus_model/model.safetensors', device='cpu')
        new_weights = {k.replace('caduceus.', ''): v for k, v in weights.items()}
        self.encoder.load_state_dict(new_weights, strict=False)
        
        # 📏 Caduceus is bidirectional, so output dim is d_model * 2 (256 * 2 = 512)
        hidden_dim = config.d_model * 2 
        
        self.classifier = nn.Sequential(
            nn.Linear(hidden_dim, 256),
            nn.ReLU(),
            nn.Dropout(0.1),
            nn.Linear(256, num_labels)
        )

    def forward(self, input_ids, labels=None):
        outputs = self.encoder(input_ids=input_ids)
        pooled = outputs.last_hidden_state.mean(dim=1) if hasattr(outputs, 'last_hidden_state') else outputs[0].mean(dim=1)
        logits = self.classifier(pooled)
        
        loss = None
        if labels is not None:
            loss = nn.CrossEntropyLoss()(logits, labels)
            
        return {'loss': loss, 'logits': logits}

# Initialize and move to GPU
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model = CaduceusClassifier(config=config, num_labels=5).to(device)

# MULTI-GPU SETUP
print("Using single GPU.")

print(f"Total parameters: {sum(p.numel() for p in model.parameters()):,}")

Using single GPU.
Total parameters: 7,857,925


In [ ]:
from torch.amp import GradScaler, autocast
from transformers import get_cosine_schedule_with_warmup
from tqdm import tqdm
import os

# Optimizer & Scheduler
optimizer = torch.optim.AdamW(model.parameters(), lr=3e-5, weight_decay=0.01)
num_epochs = 3
ACCUMULATION_STEPS = 8  # Adjust based on your new DataLoader batch size

total_steps = (len(train_loader) // ACCUMULATION_STEPS) * num_epochs
scheduler = get_cosine_schedule_with_warmup(
    optimizer, num_warmup_steps=int(0.1 * total_steps), num_training_steps=total_steps
)

# Updated AMP Scaler for PyTorch 2.10+
scaler = GradScaler('cuda')
os.makedirs('/kaggle/working/checkpoints', exist_ok=True)

print("Starting Training...")

for epoch in range(num_epochs):
    model.train()
    total_loss = 0
    optimizer.zero_grad()
    
    progress_bar = tqdm(train_loader, desc=f"Epoch {epoch+1}/{num_epochs}")
    
    for step, batch in enumerate(progress_bar):
        input_ids = batch['input_ids'].to(device)
        labels = batch['labels'].to(device)
        
        # Updated autocast syntax
        with autocast('cuda', dtype=torch.bfloat16):
            outputs = model(input_ids=input_ids, labels=labels)
            loss = outputs['loss'] / ACCUMULATION_STEPS
            
        scaler.scale(loss).backward()
        
        # Optimizer step
        if (step + 1) % ACCUMULATION_STEPS == 0 or (step + 1) == len(train_loader):
            scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
            
            scaler.step(optimizer)
            scaler.update()
            optimizer.zero_grad()
            scheduler.step()
            
        total_loss += loss.item() * ACCUMULATION_STEPS
        progress_bar.set_postfix({'loss': f"{loss.item() * ACCUMULATION_STEPS:.4f}"})
        
    avg_loss = total_loss / len(train_loader)
    print(f"Epoch {epoch+1} Average Training Loss: {avg_loss:.4f}")
    
    # Save Checkpoint (Simplified since DataParallel is removed)
    checkpoint_path = f'/kaggle/working/checkpoints/model_epoch_{epoch+1}.pth'
    torch.save(model.state_dict(), checkpoint_path)
    print(f"Checkpoint saved to {checkpoint_path}")

print("Training Complete!")

Starting Training...


Epoch 1/3:   1%|          | 99/14042 [03:43<8:03:15,  2.08s/it, loss=1.6201]